# 06: TestPyPI Integration & End-to-End Validation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Red1-Rahman/NiriZan/blob/main/experiments/06_testPYPI.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/Red1-Rahman/NiriZan/main/experiments/06_testPYPI.ipynb)

This notebook validates the `nirizan` package distribution installed directly from TestPyPI. It verifies package integrity, inspects exported API surfaces, executes a multi-span asynchronous tracing pipeline (workflow, retrieval, generation), persists trace trees to SQLite storage, and evaluates system performance via the `BehavioralAnchorMetric` visual dashboard.

### 1. Package Installation & Logging Setup
Install or upgrade `nirizan` directly from TestPyPI and activate internal log handlers to monitor runtime telemetry.

In [1]:
# 1. Install/Upgrade nirizan from TestPyPI
!pip install --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ nirizan --upgrade --quiet

import logging

# 2. Attach logging handler to display NiriZan internal logs
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    force=True
)

logger = logging.getLogger("nirizan")
logger.info("NiriZan logging engine attached and active.")

2026-08-09 08:05:07,921 [INFO] nirizan: NiriZan logging engine attached and active.


### 2. API Surface Inspection
Reflect on core modules (`tracer`, `spans`, `collector`, `trace_repository`, `behavioral_anchor`) to verify public classes, methods, and enum exports post-installation.

In [2]:
import inspect
import nirizan.instrumentation.spans as span_mod
import nirizan.instrumentation.tracer as tracer_mod
import nirizan.orchestrator.collector as collector_mod
import nirizan.storage.trace_repository as repo_mod
import nirizan.metrics.behavioral_anchor as metric_mod

api_registry = {
    "SpanKind": span_mod.SpanKind,
    "Tracer": tracer_mod.Tracer,
    "SpanHandle": tracer_mod.SpanHandle,
    "TraceCollector": collector_mod.TraceCollector,
    "SQLiteTraceRepository": repo_mod.SQLiteTraceRepository,
    "BehavioralAnchorMetric": metric_mod.BehavioralAnchorMetric,
}

print("=" * 65)
print("             NIRIZAN PACKAGE API SURFACE EXPORTS")
print("=" * 65)

for name, cls in api_registry.items():
    print(f"\n📦 Class/Enum: {name}")
    print(f"   Module: {cls.__module__}")
    print(f"   Public Attributes/Methods:")

    if hasattr(cls, "__members__"):  # Handle SpanKind Enum
        for member in cls:
            print(f"    └─ {member.name} = {member.value}")
    else:
        methods = [m for m in dir(cls) if not m.startswith("_")]
        for m in methods:
            attr = getattr(cls, m)
            sig = str(inspect.signature(attr)) if callable(attr) and not inspect.isclass(attr) else ""
            print(f"    └─ {m}{sig}")

print("=" * 65)

             NIRIZAN PACKAGE API SURFACE EXPORTS

📦 Class/Enum: SpanKind
   Module: nirizan.instrumentation.spans
   Public Attributes/Methods:
    └─ PLANNING = planning
    └─ RETRIEVAL = retrieval
    └─ TOOL_USE = tool_use
    └─ GENERATION = generation

📦 Class/Enum: Tracer
   Module: nirizan.instrumentation.tracer
   Public Attributes/Methods:
    └─ clear(self) -> None
    └─ get_assembled_trace(self, trace_id: uuid.UUID | None = None) -> nirizan.instrumentation.spans.Trace
    └─ session(self, session_id: uuid.UUID | None = None) -> AsyncGenerator[uuid.UUID, NoneType]
    └─ start_span(self, name: str, kind: nirizan.instrumentation.spans.SpanKind, attributes: dict[str, typing.Any] | None = None, input_payload: str | None = None) -> AsyncGenerator[nirizan.instrumentation.tracer.SpanHandle, NoneType]

📦 Class/Enum: SpanHandle
   Module: nirizan.instrumentation.tracer
   Public Attributes/Methods:
    └─ output_payload

📦 Class/Enum: TraceCollector
   Module: nirizan.orchestrator.

### 3. Traced Agent Pipeline Execution
Execute an asynchronous e-commerce support agent pipeline. A root parent span maintains contextual context while child spans record retrieval and generation events before persisting the trace to SQLite.

In [3]:
import asyncio
import numpy as np
from nirizan.instrumentation.spans import SpanKind
from nirizan.instrumentation.tracer import Tracer, _CURRENT_TRACE_ID
from nirizan.orchestrator.collector import TraceCollector
from nirizan.storage.trace_repository import SQLiteTraceRepository

# 1. Initialize Storage, Collector, and Tracer
repository = SQLiteTraceRepository()
collector = TraceCollector(repository=repository)
tracer = Tracer(application_name="ecommerce_support_agent")

# Dynamically select valid SpanKind enum for the root parent span
ROOT_KIND = getattr(SpanKind, "CHAIN", getattr(SpanKind, "AGENT", getattr(SpanKind, "GENERATION", list(SpanKind)[0])))

async def run_support_agent_demo():
    logger.info("Starting traced agent pipeline session...")

    root_trace_id = None

    # 2. Open Root Parent Span to establish trace context
    async with tracer.start_span(
        name="customer_support_pipeline",
        kind=ROOT_KIND,
        input_payload="User Query: Refund policy for damaged goods"
    ) as root_span:

        # Capture trace_id generated for root pipeline
        root_trace_id = _CURRENT_TRACE_ID.get()
        logger.info(f"Active Trace ID: {root_trace_id}")

        # Child Span 1: Context Retrieval
        logger.info("Executing retrieval step...")
        async with tracer.start_span(
            name="policy_vector_retrieval",
            kind=SpanKind.RETRIEVAL,
            input_payload="What is the refund policy for damaged goods?"
        ) as retrieval_span:
            await asyncio.sleep(0.04)  # Simulate DB query latency
            retrieval_span.output_payload = "Policy: Full refund within 14 days if damaged on arrival."

        # Child Span 2: Response Generation
        logger.info("Executing generation step...")
        async with tracer.start_span(
            name="llm_response_generation",
            kind=SpanKind.GENERATION,
            input_payload="User Query + Policy Context"
        ) as gen_span:
            await asyncio.sleep(0.09)  # Simulate LLM inference latency
            gen_span.output_payload = "If your item arrived damaged, you can receive a full refund within 14 days."

        root_span.output_payload = "Query handled successfully."

    # 3. Retrieve assembled trace using captured root trace_id
    assembled_trace = tracer.get_assembled_trace(trace_id=root_trace_id)

    # 4. Await async persistence & collector enqueuing
    await repository.save(assembled_trace)
    await collector.enqueue_trace(assembled_trace)

    logger.info(f"Trace saved | ID: {assembled_trace.trace_id} | Total Spans Captured: {len(assembled_trace.spans)}")
    return assembled_trace

# Run in Jupyter via top-level await
trace = await run_support_agent_demo()

2026-08-09 08:05:08,314 [INFO] nirizan: Starting traced agent pipeline session...
2026-08-09 08:05:08,315 [INFO] nirizan: Active Trace ID: c8a9d1ef-1004-4648-930f-a9b8172e4ab0
2026-08-09 08:05:08,317 [INFO] nirizan: Executing retrieval step...
2026-08-09 08:05:08,359 [INFO] nirizan: Executing generation step...
2026-08-09 08:05:08,460 [INFO] nirizan: Trace saved | ID: c8a9d1ef-1004-4648-930f-a9b8172e4ab0 | Total Spans Captured: 3


### 4. Metric Evaluation & Operational Dashboard
Load the persisted trace, run alignment scoring with `BehavioralAnchorMetric`, calculate overall operational health, and render the evaluation dashboard.

In [4]:
import inspect
from nirizan.metrics.behavioral_anchor import BehavioralAnchorMetric
from nirizan.trust.attribution import DriftAttribution
from nirizan.reporting.health_score import compute_system_health_score

# 1. Retrieve trace from SQLite storage
if inspect.iscoroutinefunction(repository.get):
    stored_trace = await repository.get(trace.trace_id)
else:
    stored_trace = repository.get(trace.trace_id)

if stored_trace is None:
    stored_trace = trace

# 2. Evaluate Trace against Behavioral Anchor Metric
logger.info("Evaluating trace against BehavioralAnchorMetric...")
target_vector = np.array([1.0, 0.0, 0.0])
metric = BehavioralAnchorMetric(
    target_embedding=target_vector,
    threshold=0.85,
    embedding_fn=lambda text: np.array([0.96, 0.04, 0.0])
)

eval_results = await metric.evaluate(stored_trace)
metric_res = eval_results[0] if isinstance(eval_results, list) and len(eval_results) > 0 else eval_results

# 3. Compute Operational System Health Score
health_score = compute_system_health_score(
    quality_score=metric_res.score if hasattr(metric_res, "score") else 0.95,
    confidence=0.98,
    attribution=DriftAttribution.NONE
)

# 4. Render Evaluation Dashboard
print("\n" + "=" * 65)
print("                  NIRIZAN EVALUATION DASHBOARD               ")
print("=" * 65)
print(f" Application        : {stored_trace.application_name}")
print(f" Trace ID           : {stored_trace.trace_id}")
print(f" Captured Spans     : {len(stored_trace.spans)}")
print("-" * 65)

for idx, span in enumerate(stored_trace.spans, start=1):
    duration = (span.ended_at - span.started_at).total_seconds() * 1000 if (span.ended_at and span.started_at) else 0.0
    print(f" Span #{idx}: [{span.kind.name}] '{span.name}'")
    print(f"   ├─ Duration : {duration:.2f} ms")
    print(f"   ├─ Input    : {span.input_payload}")
    print(f"   └─ Output   : {span.output_payload}")

print("-" * 65)
if hasattr(metric_res, "score"):
    print(f" Metric Name        : {metric_res.metric_name}")
    print(f" Alignment Score    : {metric_res.score:.4f} / 1.0000")
    print(f" Status Band        : {metric_res.details.get('band', 'N/A')}")
print("=" * 65)
print(f" 🟢 OVERALL SYSTEM HEALTH SCORE: {health_score:.1f} / 100")
print("=" * 65 + "\n")

2026-08-09 08:05:08,477 [INFO] nirizan: Evaluating trace against BehavioralAnchorMetric...



                  NIRIZAN EVALUATION DASHBOARD               
 Application        : ecommerce_support_agent
 Trace ID           : c8a9d1ef-1004-4648-930f-a9b8172e4ab0
 Captured Spans     : 3
-----------------------------------------------------------------
 Span #1: [GENERATION] 'customer_support_pipeline'
   ├─ Duration : 135.39 ms
   ├─ Input    : User Query: Refund policy for damaged goods
   └─ Output   : Query handled successfully.
 Span #2: [RETRIEVAL] 'policy_vector_retrieval'
   ├─ Duration : 40.64 ms
   ├─ Input    : What is the refund policy for damaged goods?
   └─ Output   : Policy: Full refund within 14 days if damaged on arrival.
 Span #3: [GENERATION] 'llm_response_generation'
   ├─ Duration : 90.35 ms
   ├─ Input    : User Query + Policy Context
   └─ Output   : If your item arrived damaged, you can receive a full refund within 14 days.
-----------------------------------------------------------------
 Metric Name        : behavioral_anchor
 Alignment Score    : 0.9991